**Author:** Dorys Trujillo  
**Project:** Legal Uncertainty Index (IIJ)   
**Data Source:** Ministry of Commerce, Industry and Tourism (MinCIT)  
**Period:** 2018–2025

This notebook implements an automated procedure to identify the legal provision that each draft decree aims to amend or regulate. Using text files generated through OCR, a large language model (LLM) analyzes the legal content and extracts structured information about the target regulation, the type of norm, and other relevant metadata. The workflow combines the reasoning capabilities of the model with a Retrieval-Augmented Generation (RAG) approach that prioritizes key document segments—particularly the introductory section and the operative clause introduced by the term “DECRETA”—to improve the accuracy of normative identification. The extracted results are stored in a structured dataset for subsequent validation and analysis.



In [1]:
# Dependency installation
# Installs the OpenAI client library (compatible with DeepSeek API) and pandas for data handling.

#!pip install openai pandas

In [ ]:
from google.colab import userdata
from openai import OpenAI

# Retrieve the API key stored securely in the Colab environment.
api_key = userdata.get('DEEPSEEK_API_KEY')

# Initialize the OpenAI client configured to communicate with the DeepSeek API endpoint.
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

# Confirm that the API client has been successfully initialized.
print("Connection successfully established")

In [ ]:
# Mount Google Drive to access project files stored in the user's Drive.
from google.colab import drive
import os
drive.mount('/content/drive')

In [ ]:
# Import required libraries for file management, data processing, API interaction,
# JSON parsing, and execution control during the document processing pipeline.

import os
import pandas as pd
from openai import OpenAI
from google.colab import userdata
import json
import time

In [ ]:
# Configuration
# Initializes the API client and defines the processing scope (years and base directory).
client = OpenAI(api_key=userdata.get('DEEPSEEK_API_KEY'), base_url="https://api.deepseek.com")
RUTA_BASE = '/content/drive/MyDrive/tesis/pdf_ocr'
ANIOS_PROCESAR = [str(a) for a in range(2018, 2026)]

# Stores all extracted records across years to produce a single consolidated output.
resultados = []

def _safe_strip_json(texto: str) -> str:
    """Removes common Markdown wrappers to keep a valid JSON payload only."""
    if not isinstance(texto, str):
        return ""
    t = texto.strip()
    if "```" in t:
        if "```json" in t:
            t = t.split("```json", 1)[1]
            t = t.split("```", 1)[0]
        else:
            t = t.split("```", 1)[1]
            t = t.split("```", 1)[0]
    return t.strip()

def _extract_key_windows(full_text: str, max_total: int = 4500) -> str:
    """
    Builds a compact analysis input emphasizing:
    (i) the opening section (title/object), and
    (ii) the resolutive segment around 'DECRETA' (primary legal operative text).
    """
    if not full_text:
        return ""
    txt = full_text.replace("\x00", " ")
    head = txt[:1200]

    # Locate the resolutive section marker (robust to OCR spacing and minor variants).
    upper = txt.upper()
    idx = upper.find("DECRETA")
    if idx == -1:
        idx = upper.find("D E C R E T A")
    if idx == -1:
        idx = upper.find("DECRET A")
    if idx == -1:
        idx = upper.find("DECRET")

    if idx != -1:
        start = max(0, idx - 400)
        end = min(len(txt), idx + 2000)
        decreta_win = txt[start:end]
    else:
        decreta_win = txt[1200:3200]

    combined = (head + "\n\n---\n\n" + decreta_win).strip()
    return combined[:max_total]

# Year-level loop (processes all years in the defined range).
for ANIO_PROCESAR in ANIOS_PROCESAR:

    ruta_txt = os.path.join(RUTA_BASE, ANIO_PROCESAR, 'txt')
    print(f"Iniciando extracción avanzada para el año {ANIO_PROCESAR}...")

    # Skips missing directories to avoid interrupting the batch run.
    if not os.path.exists(ruta_txt):
        print(f"⚠️ No existe carpeta: {ruta_txt} (se omite {ANIO_PROCESAR})")
        continue

    # Collects all text files produced by OCR for the current year.
    archivos = [f for f in os.listdir(ruta_txt) if f.endswith('.txt')]

    for nombre_archivo in archivos:
        try:
            with open(os.path.join(ruta_txt, nombre_archivo), 'r', encoding='utf-8') as f:
                # Reads a sufficiently large prefix to capture both the preamble and the resolutive section.
                full_text = f.read(12000)

            # Constructs a focused excerpt to reduce noise from recitals while preserving operative clauses.
            texto_analisis = _extract_key_windows(full_text, max_total=4500)

            # Prompt in Spanish is intentionally preserved to match the legal domain and corpus language.
            prompt = f"""Analiza este documento legal colombiano y determina QUÉ norma principal modifica o reglamenta.

OBJETIVO:
- Identificar la "norma_objetivo" real (Decreto, Ley, Estatuto, Arancel, etc.) que el proyecto de decreto modifica o reglamenta.

MÉTODO (obligatorio):
A) Compara el OBJETO del inicio (primer párrafo/título) con lo que aparece en la parte resolutiva, especialmente lo que sigue a la palabra "DECRETA".
B) PRIORIZA SIEMPRE la parte resolutiva sobre los considerandos.

JERARQUÍA DE DECISIÓN (prioridad 1 a 4):
1) MODIFICACIÓN DIRECTA (resolutiva): Si después de "DECRETA" aparece explícitamente "Modifíquese / Adiciónese / Sustitúyase / Deróguese" y se menciona "Decreto X", "Ley Y", "Estatuto Tributario", "Estatuto Aduanero", "Arancel de Aduanas", esa es la norma_objetivo.
2) REGLAMENTACIÓN: Si la resolutiva indica "reglamentar" una Ley/Estatuto/Arancel, esa es la norma_objetivo.
3) COMPILATORIO / ÚNICO: Si crea un decreto "único reglamentario" o compila normas (poco común), identifica ese objetivo.
4) NORMA NUEVA (MATERIA): Si NO modifica ni reglamenta explícitamente una norma previa en la resolutiva, entonces:
   - tipo_norma = "Materia Técnica"
   - norma_objetivo = el TEMA principal del título/objeto (3–6 palabras)
   - es_modificatoria = "No"
   IMPORTANTE: ignora decretos/leyes citados SOLO en considerandos como antecedentes.

REGLAS DE EXTRACCIÓN:
- Si mencionan "Estatuto Tributario" o "Decreto 624 de 1989", norma_objetivo debe ser "Estatuto Tributario (D. 624/1989)" cuando aplique.
- Si mencionan "Estatuto Aduanero" o "Decreto 1165 de 2019" (u otro), regístralo con número y año si aparece.
- Si mencionan "Arancel de Aduanas" o su decreto base (si se menciona), usa "Arancel de Aduanas" como norma_objetivo.
- entidad_emisora: ministerio/entidad que emite (si no aparece, pon "No identificado").
- articulo_especifico: si se menciona artículo(s) afectado(s) en la resolutiva, inclúyelo; si no, "No aplica".
- resumen_breve: máximo 15 palabras.

Responde estrictamente en JSON (sin texto adicional):
{{
  "norma_objetivo": "",
  "tipo_norma": "Decreto / Estatuto / Ley / Arancel / Materia Técnica",
  "materia_o_asunto": "",
  "entidad_emisora": "",
  "es_modificatoria": "Si/No",
  "articulo_especifico": "",
  "resumen_breve": ""
}}

TEXTO PARA ANALIZAR (incluye inicio + ventana alrededor de DECRETA):
{texto_analisis}
"""

            # Calls the LLM with constrained stochasticity to stabilize structured extraction.
            response = client.chat.completions.create(
                model="deepseek-reasoner",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5
            )

            # Parses model output as JSON after removing potential formatting wrappers.
            contenido_crudo = response.choices[0].message.content
            contenido_crudo = _safe_strip_json(contenido_crudo)
            datos_json = json.loads(contenido_crudo)

            # Adds provenance metadata for downstream auditing and traceability.
            datos_json['archivo_pdf'] = nombre_archivo.replace('.txt', '.pdf')
            datos_json['año_doc'] = ANIO_PROCESAR

            # Derives an organization label for downstream file structuring and review workflows.
            try:
                if str(datos_json.get('es_modificatoria', '')).strip().lower() == "no":
                    datos_json['carpeta_organizacion'] = datos_json.get('materia_o_asunto', '') or datos_json.get('norma_objetivo', '')
                else:
                    datos_json['carpeta_organizacion'] = datos_json.get('norma_objetivo', '') or datos_json.get('materia_o_asunto', '')
            except:
                datos_json['carpeta_organizacion'] = datos_json.get('norma_objetivo', '')

            resultados.append(datos_json)
            print(f"Procesado: {ANIO_PROCESAR}/{nombre_archivo} -> {datos_json.get('carpeta_organizacion','')}")

        except Exception as e:
            # Captures per-file failures without interrupting the batch.
            print(f"Error en {ANIO_PROCESAR}/{nombre_archivo}: {str(e)}")

        # Basic throttling to reduce the probability of API rate-limit errors.
        time.sleep(1.5)

# Consolidated CSV output across all processed years.
df_final = pd.DataFrame(resultados)

# Uses a fixed output name for reproducibility and consistent downstream references.
nombre_csv = "mapeo_avanzado_2018_2025.csv"

df_final.to_csv(nombre_csv, index=False, encoding='utf-8-sig', sep=";")
print(f"\n Proceso terminado. Archivo '{nombre_csv}' listo para revisión (incluye 'carpeta_organizacion').")